# import libraries

In [49]:
import geopandas as gpd
import numpy as np
import pandas as pd
import os
from pathlib import Path
import re
from shapely.geometry import MultiPolygon
import warnings

# variables

In [50]:
ROOT_DIR = Path("_data/eurocrops2")
OUT_FOLDER = "_data/exports/country_year_exports"

CROPS_SIZE = 500

# create datasets

## read datasets

In [51]:
def get_inventory(root_path):
    inventory = []
    
    for country_dir in root_path.iterdir():
        if country_dir.is_dir():
            for file_path in country_dir.glob("*.parquet"):
                parts = file_path.stem.split('_')
                
                country_id = parts[0]   
                year = int(parts[-1])   
                country = country_dir.name  
                
                inventory.append({"country": country, 
                                  "sub_region": country_id, 
                                  "year": year, 
                                  "id": f"{country}_{year}", 
                                  "sub_region_id": f"{country_id}_{year}", 
                                  "path": file_path})
    
    df = pd.DataFrame(inventory)
    
    if not df.empty:
        df = df.sort_values(by=["country", "year"]).reset_index(drop=True)
        
    return df

In [52]:
available_datasets = get_inventory(ROOT_DIR)
available_datasets

,country,sub_region,year,id,sub_region_id,path
0,at,at,2017,at_2017,at_2017,_data/eurocrops2/at/at_2017.parquet
1,at,at,2018,at_2018,at_2018,_data/eurocrops2/at/at_2018.parquet
2,at,at,2019,at_2019,at_2019,_data/eurocrops2/at/at_2019.parquet
3,be,be2,2016,be_2016,be2_2016,_data/eurocrops2/be/be2_2016.parquet
4,be,be3,2016,be_2016,be3_2016,_data/eurocrops2/be/be3_2016.parquet
5,be,be2,2018,be_2018,be2_2018,_data/eurocrops2/be/be2_2018.parquet
6,be,be3,2018,be_2018,be3_2018,_data/eurocrops2/be/be3_2018.parquet
7,be,be3,2019,be_2019,be3_2019,_data/eurocrops2/be/be3_2019.parquet
8,be,be2,2019,be_2019,be2_2019,_data/eurocrops2/be/be2_2019.parquet
9,bg,bg,2016,bg_2016,bg_2016,_data/eurocrops2/bg/bg_2016.parquet


## select columns of interest

In [53]:
# # see what columns we have on the datasets
# gpd.read_parquet(available_datasets['path'][0]).head(3)

In [54]:
COLUMNS = ['original_code', 'geometry', 'cropfield']

In [55]:
def load_and_filter_datasets(dataframe, columns_of_interest):
    processed = {}
    
    for _, row in dataframe.iterrows():
        gdf = gpd.read_parquet(row['path'])
        processed[row['sub_region_id']] = gdf[columns_of_interest].copy()
        
    return processed

In [56]:
datasets = load_and_filter_datasets(available_datasets, COLUMNS)
datasets.keys()

dict_keys(['at_2017', 'at_2018', 'at_2019', 'be2_2016', 'be3_2016', 'be2_2018', 'be3_2018', 'be3_2019', 'be2_2019', 'bg_2016', 'bg_2018', 'bg_2019'])

In [57]:
datasets['at_2019'].head(2)

,original_code,geometry,cropfield
0,701,"MULTIPOLYGON (((4700031.26 2819279.166, 470001...",73516
1,710,"MULTIPOLYGON (((4632700.851 2784799.389, 46327...",794497


## mapping the crops on the dataset

In [58]:
TARGET_COLUMNS = ['original_code', 'geometry', 'hcat4_code', 'hcat4_name', 'cropfield']

In [59]:
def load_and_map_datasets(inventory_df, mapping_csv_path, columns_of_interest):
    mapping_df = pd.read_csv(mapping_csv_path)

    mapping_df["original_code"] = mapping_df["original_code"].astype(str).str.strip()
    mapping_df["nuts"] = mapping_df["nuts"].astype(str).str.strip()

    processed = {}

    for _, row in inventory_df.iterrows():
        print(f"Processing {row['sub_region_id']}")
        gdf = gpd.read_parquet(row["path"])

        gdf["original_code"] = gdf["original_code"].astype(str).str.strip()

        country_map = mapping_df[mapping_df["nuts"] == row["sub_region"]]
        if country_map.empty:
            country_map = mapping_df[mapping_df["nuts"] == row["country"]]

        merged_gdf = gdf.merge(country_map[["original_code", 
                                            "hcat4_code", 
                                            "hcat4_name"]], 
                                            on="original_code", 
                                            how="left")
        
        final_cols = [c for c in columns_of_interest if c in merged_gdf.columns]
        processed[row["sub_region_id"]] = merged_gdf[final_cols].copy()
        
    return processed

In [60]:
datasets['at_2019']

,original_code,geometry,cropfield
0,701,"MULTIPOLYGON (((4700031.26 2819279.166, 470001...",73516
1,710,"MULTIPOLYGON (((4632700.851 2784799.389, 46327...",794497
2,353,"MULTIPOLYGON (((4663344.099 2825196.383, 46633...",114671
3,307,"MULTIPOLYGON (((4841746.513 2757224.136, 48417...",3541432
4,901,"MULTIPOLYGON (((4746671.669 2832442.896, 47466...",209662
...,...,...,...
2529882,721,"MULTIPOLYGON (((4665818.986 2786144.78, 466583...",3698460
2529883,717,"MULTIPOLYGON (((4319046.913 2656793.229, 43190...",3879062
2529884,717,"MULTIPOLYGON (((4678022.147 2779845.087, 46780...",4029241
2529885,716,"MULTIPOLYGON (((4575907.161 2807778.711, 45759...",4007983


In [61]:
datasets = load_and_map_datasets(available_datasets, "_data/eurocrops2/eurocrops.csv", TARGET_COLUMNS)
# datasets.keys()

Processing at_2017
Processing at_2018
Processing at_2019
Processing be2_2016
Processing be3_2016
Processing be2_2018
Processing be3_2018
Processing be3_2019
Processing be2_2019
Processing bg_2016
Processing bg_2018
Processing bg_2019


In [62]:
datasets['at_2018'].head(3)

,original_code,geometry,hcat4_code,hcat4_name,cropfield
0,716,"MULTIPOLYGON (((4673153.335 2812330.209, 46731...",3320100000,permanent_grassland,7304
1,351,"MULTIPOLYGON (((4673446.689 2812521.396, 46734...",3360000000,tree_wood_forest,7305
2,717,"MULTIPOLYGON (((4673351.203 2812466.686, 46733...",3320100000,permanent_grassland,7306


## eliminate Nan values on geometry

In [63]:
def eliminate_Nan(datasets):
    cleaned_datasets = {}
    
    with warnings.catch_warnings():
        warnings.filterwarnings('ignore', 'GeoSeries.notna', UserWarning)
        
        for key in datasets.keys():
            gdf = datasets[key]
            
            missing_count = gdf.geometry.isna().sum()
            empty_count = gdf.geometry.is_empty.sum()
            
            if missing_count > 0 or empty_count > 0:
                print(f"Cleaning '{key}': Removed {missing_count} NaN and {empty_count} Empty.")
            
            cleaned_gdf = gdf[gdf.geometry.notna() & (~gdf.geometry.is_empty)].copy()
            cleaned_datasets[key] = cleaned_gdf
            
    return cleaned_datasets

In [64]:
datasets = eliminate_Nan(datasets)

Cleaning 'at_2017': Removed 0 NaN and 14 Empty.
Cleaning 'at_2018': Removed 0 NaN and 13 Empty.
Cleaning 'at_2019': Removed 0 NaN and 3 Empty.
Cleaning 'be2_2016': Removed 0 NaN and 21 Empty.
Cleaning 'be2_2018': Removed 0 NaN and 8 Empty.
Cleaning 'be2_2019': Removed 0 NaN and 2 Empty.
Cleaning 'bg_2016': Removed 0 NaN and 1195 Empty.
Cleaning 'bg_2018': Removed 0 NaN and 1489 Empty.
Cleaning 'bg_2019': Removed 0 NaN and 1517 Empty.


## select crops of interest

winter_barley

potatoes

maize_corn_popcorn 

spring barley

clover

In [65]:
# # count the most occurent classes
# print(f"Most common classes: {datasets["dk_2018"]['hcat4_name'].value_counts()[:30]} \n")

In [66]:
# define crops of interest
crops = ['winter_barley', 'potatoes', 'maize_corn_popcorn', 'spring_barley', 'clover']

def interest_crop(gdf, crops_list):
    result = gdf[gdf['hcat4_name'].isin(crops_list)]
    return result

crops_dataset = {}

for dataset_id, gdf in datasets.items():
    subset_gdf = interest_crop(gdf, crops)
    
    if not subset_gdf.empty:
        crops_dataset[dataset_id] = subset_gdf

In [67]:
print(f"Number of old datasets: {len(datasets)}")
print(f"Old dataset keys: {datasets.keys()}")
print(f"Number of new datasets created: {len(crops_dataset)}")
print(f"New datasets created: {crops_dataset.keys()}")

Number of old datasets: 12
Old dataset keys: dict_keys(['at_2017', 'at_2018', 'at_2019', 'be2_2016', 'be3_2016', 'be2_2018', 'be3_2018', 'be3_2019', 'be2_2019', 'bg_2016', 'bg_2018', 'bg_2019'])
Number of new datasets created: 12
New datasets created: dict_keys(['at_2017', 'at_2018', 'at_2019', 'be2_2016', 'be3_2016', 'be2_2018', 'be3_2018', 'be3_2019', 'be2_2019', 'bg_2016', 'bg_2018', 'bg_2019'])


In [68]:
crops_dataset['bg_2018'].head(3)

,original_code,geometry,hcat4_code,hcat4_name,cropfield
3,10259,"MULTIPOLYGON (((5486165.628 2383185.49, 548621...",3310106000,maize_corn_popcorn,3764
17,10141,"MULTIPOLYGON (((5730114.266 2324615.744, 57301...",3310104001,winter_barley,5027
23,10259,"MULTIPOLYGON (((5758113.795 2469557.655, 57582...",3310106000,maize_corn_popcorn,9115


## merging back sub regions and add year and country id

In [69]:
target_countries = available_datasets['country'].unique()
target_countries

array(['at', 'be', 'bg'], dtype=object)

In [70]:
def merge_sub_regions(dataset_dict, countries=['be']):
    if isinstance(countries, str):
        countries = [countries]
        
    merged_results = {}

    for country in countries:
        pattern = rf'(^|_)({country}\d+)_' 
        
        country_keys = [k for k in dataset_dict.keys() if k.startswith(f"{country}")]
        
        groups = {}
        
        for k in country_keys:
            parts = k.split('_')
            year = parts[-1]
            
            new_key = f"{country}_{year}"
            
            df_temp = dataset_dict[k].copy()
            df_temp['country_id'] = country
            df_temp['year'] = year
            
            if new_key not in groups:
                groups[new_key] = []
            groups[new_key].append(df_temp)
            
        for target_name, df_list in groups.items():
            if len(df_list) > 1:
                merged_results[target_name] = pd.concat(df_list, ignore_index=True)
            else:
                merged_results[target_name] = df_list[0]
            
    return merged_results

In [71]:
crops_dataset = merge_sub_regions(crops_dataset, countries=target_countries)
crops_dataset.keys()

dict_keys(['at_2017', 'at_2018', 'at_2019', 'be_2016', 'be_2018', 'be_2019', 'bg_2016', 'bg_2018', 'bg_2019'])

In [72]:
crops_dataset['bg_2018'].head(3)

,original_code,geometry,hcat4_code,hcat4_name,cropfield,country_id,year
3,10259,"MULTIPOLYGON (((5486165.628 2383185.49, 548621...",3310106000,maize_corn_popcorn,3764,bg,2018
17,10141,"MULTIPOLYGON (((5730114.266 2324615.744, 57301...",3310104001,winter_barley,5027,bg,2018
23,10259,"MULTIPOLYGON (((5758113.795 2469557.655, 57582...",3310106000,maize_corn_popcorn,9115,bg,2018


## eliminate multipoligons with more than one poligon

In [73]:
def drop_multipoligons(datasets):
    cleaned_results = {}
    
    for name, gdf in datasets.items():
        multi_parts_indices = []
        eliminates = 0
        
        for idx, geom in gdf['geometry'].items():
            if isinstance(geom, MultiPolygon) and len(geom.geoms) > 1:
                multi_parts_indices.append(idx)
                eliminates = eliminates + 1
        
        print(f"Eliminating multipoligon {eliminates} poligons in '{name}'.")
        
        if multi_parts_indices:
            cleaned_gdf = gdf.drop(index=multi_parts_indices)
            cleaned_results[name] = cleaned_gdf.reset_index(drop=True)
        else:
            cleaned_results[name] = gdf.copy()
            
    return cleaned_results

In [74]:
crops_dataset = drop_multipoligons(crops_dataset)

Eliminating multipoligon 8 poligons in 'at_2017'.
Eliminating multipoligon 1 poligons in 'at_2018'.
Eliminating multipoligon 7 poligons in 'at_2019'.
Eliminating multipoligon 122 poligons in 'be_2016'.
Eliminating multipoligon 88 poligons in 'be_2018'.
Eliminating multipoligon 93 poligons in 'be_2019'.
Eliminating multipoligon 294 poligons in 'bg_2016'.
Eliminating multipoligon 285 poligons in 'bg_2018'.
Eliminating multipoligon 345 poligons in 'bg_2019'.


In [75]:
def drop_small_polygons(datasets, min_area_m2=10):
    cleaned_results = {}
    
    for name, gdf in datasets.items():
        if gdf.empty:
            cleaned_results[name] = gdf
            continue
            
        temp_areas = gdf.geometry.to_crs(epsg=3857).area
        
        small_polygons_mask = temp_areas < min_area_m2
        small_indices = gdf.index[small_polygons_mask].tolist()
        
        num_eliminated = len(small_indices)
        
        if num_eliminated > 0:
            print(f"Eliminating {num_eliminated} polygons in '{name}' smaller than {min_area_m2}m2.")
            print(f"Eliminated row indices: {small_indices}")
            
            cleaned_gdf = gdf.drop(index=small_indices).reset_index(drop=True)
            cleaned_results[name] = cleaned_gdf
        else:
            print(f"No small polygons found in '{name}'.")
            cleaned_results[name] = gdf.copy()
            
    return cleaned_results

In [76]:
crops_dataset = drop_small_polygons(crops_dataset, min_area_m2=20)

No small polygons found in 'at_2017'.
Eliminating 4 polygons in 'at_2018' smaller than 20m2.
Eliminated row indices: [32204, 226737, 313260, 313479]
Eliminating 2 polygons in 'at_2019' smaller than 20m2.
Eliminated row indices: [107103, 318798]
Eliminating 2 polygons in 'be_2016' smaller than 20m2.
Eliminated row indices: [27042, 104071]
No small polygons found in 'be_2018'.
No small polygons found in 'be_2019'.
Eliminating 120 polygons in 'bg_2016' smaller than 20m2.
Eliminated row indices: [291, 1912, 2496, 3659, 4342, 4351, 4744, 5642, 6999, 7089, 7772, 9173, 10138, 12132, 13047, 13875, 16219, 16344, 17221, 17844, 18938, 19132, 19989, 21056, 22101, 23704, 24868, 25148, 30020, 30766, 31420, 33053, 33078, 33433, 33739, 34113, 34257, 34951, 34961, 34979, 36630, 37751, 38206, 39873, 41235, 41369, 42917, 43295, 45139, 45143, 46486, 48801, 48949, 51468, 51650, 54044, 54923, 55005, 55539, 57444, 57878, 58893, 59061, 60321, 61089, 62006, 62366, 62652, 63217, 63985, 64443, 64516, 64558, 6528

In [77]:
crops_dataset['bg_2018'].head(3)

,original_code,geometry,hcat4_code,hcat4_name,cropfield,country_id,year
0,10259,"MULTIPOLYGON (((5486165.628 2383185.49, 548621...",3310106000,maize_corn_popcorn,3764,bg,2018
1,10141,"MULTIPOLYGON (((5730114.266 2324615.744, 57301...",3310104001,winter_barley,5027,bg,2018
2,10259,"MULTIPOLYGON (((5758113.795 2469557.655, 57582...",3310106000,maize_corn_popcorn,9115,bg,2018


## limit datasets to a certain number of rows

In [86]:
limited_crops_dataset = {}

for name, gdf in crops_dataset.items():
    if gdf.empty:
        limited_crops_dataset[name] = gdf
        continue

    # We keep include_groups=True (default) to keep the column, 
    # but we use group_keys=False to avoid the multi-index warning.
    limited_gdf = gdf.groupby('hcat4_name', group_keys=False).apply(
        lambda x: x.sample(n=min(len(x), CROPS_SIZE), random_state=42)
    ).reset_index(drop=True)

    # print(f"\n--- Dataset: {name} ---")
    # print(limited_gdf['hcat4_name'].value_counts())
    
    limited_crops_dataset[name] = limited_gdf

/var/folders/m1/jbwb10r115nghdm3qdhnjtnm0000gn/T/ipykernel_93461/3886380520.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  limited_gdf = gdf.groupby('hcat4_name', group_keys=False).apply(
/var/folders/m1/jbwb10r115nghdm3qdhnjtnm0000gn/T/ipykernel_93461/3886380520.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  limited_gdf = gdf.groupby('hcat4_name', group_keys=False).apply(
/var/folders/m1/jbwb10r115nghdm3

In [87]:
limited_crops_dataset['bg_2019']

,original_code,geometry,hcat4_code,hcat4_name,cropfield,country_id,year
0,10106,"MULTIPOLYGON (((5574030.213 2219184.633, 55741...",3310202030,clover,104020,bg,2019
1,10106,"MULTIPOLYGON (((5600361.652 2175896.597, 56003...",3310202030,clover,989179,bg,2019
2,10106,"MULTIPOLYGON (((5628110.59 2176156.375, 562811...",3310202030,clover,161167,bg,2019
3,10106,"MULTIPOLYGON (((5622508.706 2197075.642, 56225...",3310202030,clover,410597,bg,2019
4,10106,"MULTIPOLYGON (((5600658.775 2174766.882, 56006...",3310202030,clover,989752,bg,2019
...,...,...,...,...,...,...,...
2495,10141,"MULTIPOLYGON (((5479306.457 2399802.659, 54793...",3310104001,winter_barley,119288,bg,2019
2496,10141,"MULTIPOLYGON (((5743773.585 2335681.56, 574380...",3310104001,winter_barley,452707,bg,2019
2497,10141,"MULTIPOLYGON (((5510748.697 2228654.406, 55107...",3310104001,winter_barley,287453,bg,2019
2498,10141,"MULTIPOLYGON (((5727152.026 2401526.782, 57271...",3310104001,winter_barley,173870,bg,2019


# get centroid and long and lat

In [89]:
def access_coords(gpd, index):
    poligon = gpd.geometry.iloc[index]

    coords = list(poligon.exterior.coords)

    coords_numpy = np.array(poligon.exterior.coords)
    return poligon, coords, coords_numpy

In [90]:
def get_centroid(gdf, centroid_as_main=False):
    gdf = gdf.copy()
    local_utm = gdf.estimate_utm_crs()
    gdf['centroid'] = gdf.geometry.to_crs(local_utm).centroid.to_crs(epsg=4326)
    gdf = gdf.to_crs(epsg=4326)
    
    if centroid_as_main: 
        gdf = gdf.set_geometry('centroid')
        
    return gdf

In [91]:
def get_long_lat(gdf):
    gdf['long'] = gdf['centroid'].x
    gdf['lat'] = gdf['centroid'].y
    return gdf

In [92]:
# check before applying transformations
print(limited_crops_dataset['bg_2018'].crs)
limited_crops_dataset['bg_2018'].head(3)

{"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "ProjectedCRS", "name": "ETRS89-extended / LAEA Europe", "base_crs": {"name": "ETRS89", "datum_ensemble": {"name": "European Terrestrial Reference System 1989 ensemble", "members": [{"name": "European Terrestrial Reference Frame 1989"}, {"name": "European Terrestrial Reference Frame 1990"}, {"name": "European Terrestrial Reference Frame 1991"}, {"name": "European Terrestrial Reference Frame 1992"}, {"name": "European Terrestrial Reference Frame 1993"}, {"name": "European Terrestrial Reference Frame 1994"}, {"name": "European Terrestrial Reference Frame 1996"}, {"name": "European Terrestrial Reference Frame 1997"}, {"name": "European Terrestrial Reference Frame 2000"}, {"name": "European Terrestrial Reference Frame 2005"}, {"name": "European Terrestrial Reference Frame 2014"}], "ellipsoid": {"name": "GRS 1980", "semi_major_axis": 6378137, "inverse_flattening": 298.257222101}, "accuracy": "0.1", "id": {"authority":

,original_code,geometry,hcat4_code,hcat4_name,cropfield,country_id,year
0,10106,"MULTIPOLYGON (((5474933.65 2285634.099, 547493...",3310202030,clover,584902,bg,2018
1,10106,"MULTIPOLYGON (((5472427.325 2268086.729, 54724...",3310202030,clover,720683,bg,2018
2,10106,"MULTIPOLYGON (((5397692.892 2201052.955, 53977...",3310202030,clover,316076,bg,2018


In [93]:
for i in limited_crops_dataset:
    print(f"Apply functions on {i}")
    limited_crops_dataset[i] = get_centroid(limited_crops_dataset[i])
    limited_crops_dataset[i] = get_long_lat(limited_crops_dataset[i])

Apply functions on at_2017
Apply functions on at_2018
Apply functions on at_2019
Apply functions on be_2016
Apply functions on be_2018
Apply functions on be_2019
Apply functions on bg_2016
Apply functions on bg_2018
Apply functions on bg_2019


In [94]:
# check after applying transformations
print(limited_crops_dataset['bg_2018'].crs)
limited_crops_dataset['bg_2018'].head(3)

EPSG:4326


,original_code,geometry,hcat4_code,hcat4_name,cropfield,country_id,year,centroid,long,lat
0,10106,"MULTIPOLYGON (((24.13296 42.69734, 24.13301 42...",3310202030,clover,584902,bg,2018,POINT (24.13352 42.69754),24.133524,42.697536
1,10106,"MULTIPOLYGON (((24.0657 42.54625, 24.06576 42....",3310202030,clover,720683,bg,2018,POINT (24.06753 42.54695),24.067533,42.546948
2,10106,"MULTIPOLYGON (((23.03759 42.07277, 23.03776 42...",3310202030,clover,316076,bg,2018,POINT (23.03787 42.0727),23.037868,42.072705


# export data

In [95]:
if not os.path.exists(OUT_FOLDER):
    os.makedirs(OUT_FOLDER)
    print(f"Created directory: {OUT_FOLDER}")

for name in limited_crops_dataset.keys():
    temp_df = pd.DataFrame(limited_crops_dataset[name][['country_id', 'year', 'hcat4_name', 'long', 'lat', 'cropfield']]).copy()
    
    file_path = f"{OUT_FOLDER}/{name}.csv"
    
    temp_df.to_csv(file_path, index=False)
    print(f"Exported: {file_path}")

Exported: _data/exports/country_year_exports/at_2017.csv
Exported: _data/exports/country_year_exports/at_2018.csv
Exported: _data/exports/country_year_exports/at_2019.csv
Exported: _data/exports/country_year_exports/be_2016.csv
Exported: _data/exports/country_year_exports/be_2018.csv
Exported: _data/exports/country_year_exports/be_2019.csv
Exported: _data/exports/country_year_exports/bg_2016.csv
Exported: _data/exports/country_year_exports/bg_2018.csv
Exported: _data/exports/country_year_exports/bg_2019.csv
